In [1]:
# Libraries
import pandas as pd
import scipy.stats as st

In [2]:
# Load data
df = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/pokemon.csv")

## Dragon vs Grass HP

In [3]:
# Dragon vs Grass HP
dragon_hp = df[df["Type 1"] == "Dragon"]["HP"]
grass_hp = df[df["Type 1"] == "Grass"]["HP"]

In [6]:
# Two-sample t-test (two-sided by default)
t_stat, p_val_two_sided = st.ttest_ind(dragon_hp, grass_hp, equal_var=False)

# Convert to one-sided p-value safely - whn the hypothesis is directional
if t_stat > 0:
    p_val_one_sided = p_val_two_sided / 2
else:
    p_val_one_sided = 1  # t_stat in opposite direction

print("T-statistic:", t_stat)
print("One-sided P-value:", p_val_one_sided)

T-statistic: 3.3349632905124063
One-sided P-value: 0.0007993609745420598


Since 0.0008 < 0.05:

We reject H0.

There is strong statistical evidence that Dragon Pokémon have higher HP than Grass Pokémon, at the 5% significance level.

## Legendary vs Non-Legendary Stats

In [7]:
stats = ["HP", "Attack", "Defense", "Sp. Atk", "Sp. Def", "Speed"]

legendary = df[df["Legendary"] == True]
non_legendary = df[df["Legendary"] == False]

for stat in stats:
    t_stat, p_val = st.ttest_ind(legendary[stat], non_legendary[stat], equal_var=False)
    print(f"{stat} -> T-stat: {t_stat:.2f}, P-val: {p_val:.4f}")


HP -> T-stat: 8.98, P-val: 0.0000
Attack -> T-stat: 10.44, P-val: 0.0000
Defense -> T-stat: 7.64, P-val: 0.0000
Sp. Atk -> T-stat: 13.42, P-val: 0.0000
Sp. Def -> T-stat: 10.02, P-val: 0.0000
Speed -> T-stat: 11.48, P-val: 0.0000


H0 (null hypothesis): Legendary and Non-Legendary Pokémon have the same mean for the stat.

H1 (alternative hypothesis): Legendary and Non-Legendary Pokémon have different means for the stat.
###########
For all stats, P-val = 0.0000 → much smaller than 0.05.

Rule:

p < 0.05 → Reject H0

p ≥ 0.05 → Fail to reject H0

All your p-values are tiny → Reject H0 for all stats.

## Challenge 2: California Housing

In [10]:
import pandas as pd
import numpy as np
import scipy.stats as st

# Load dataset
df2 = pd.read_csv("https://raw.githubusercontent.com/data-bootcamp-v4/data/main/california_housing.csv")

# Check columns
print(df2.columns)

Index(['longitude', 'latitude', 'housing_median_age', 'total_rooms',
       'total_bedrooms', 'population', 'households', 'median_income',
       'median_house_value'],
      dtype='object')


In [11]:
# Coordinates
school = (-118, 34)
hospital = (-122, 37)

# Euclidean distance function
def distance(x1, y1, x2, y2):
    return np.sqrt((x1-x2)**2 + (y1-y2)**2)

# Distances
df2["dist_school"] = distance(df2["longitude"], df2["latitude"], school[0], school[1])
df2["dist_hospital"] = distance(df2["longitude"], df2["latitude"], hospital[0], hospital[1])

# Close = distance < 0.5 to either school or hospital
df2["close"] = np.where((df2["dist_school"] < 0.5) | (df2["dist_hospital"] < 0.5), 1, 0)


In [12]:
# split dataset into close and far houses
close_houses = df2[df2["close"] == 1]["median_house_value"]
far_houses = df2[df2["close"] == 0]["median_house_value"]


In [13]:
#Perform one-sided t-test (close houses > far houses)
t_stat, p_val_two_sided = st.ttest_ind(close_houses, far_houses, equal_var=False)

# One-sided p-value
if t_stat > 0:
    p_val_one_sided = p_val_two_sided / 2
else:
    p_val_one_sided = 1  # opposite direction

print("T-statistic:", t_stat)
print("One-sided P-value:", p_val_one_sided)


T-statistic: 37.992330214201516
One-sided P-value: 1.5032478884296307e-301


In [14]:
# interpretation of the result

if p_val_one_sided < 0.05:
    print("Reject H0: Houses close to a school or hospital are significantly more expensive.")
else:
    print("Fail to reject H0: No significant evidence that proximity increases house value.")


Reject H0: Houses close to a school or hospital are significantly more expensive.


# I reject the null hypothesis at the 5% significance level. There is strong statistical evidence that houses located near a school or hospital (within 0.5 units distance) are significantly more expensive than those farther away.